## Step 1 — ติดตั้ง Neo4j Python Driver

In [ ]:
!pip install -q neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 9.8 MB/s eta 0:00:00


## Step 2 — Import Library

In [ ]:
import neo4j
import pandas as pd

print("Neo4j Python Driver Version:", neo4j.__version__)


Neo4j Python Driver Version: 6.3.1


## Step 3 — เชื่อมต่อ Neo4j Aura

ใช้ URI จากหน้า Connect ของ Neo4j Aura แทนการเชื่อมต่อ Neo4j Desktop

In [ ]:
from getpass import getpass
from neo4j import GraphDatabase

# เปลี่ยนเป็น URI ของ Neo4j Aura ของตัวเอง
URI = "neo4j+s://c343b019.databases.neo4j.io"
USERNAME = "neo4j"
PASSWORD = getpass("Password: ")

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()
print("Connected successfully")


Password: ··········
Connected successfully


## Step 4 — หา Home Database

In [ ]:
result = driver.execute_query("SHOW HOME DATABASE")

for record in result.records:
    print(record.data())


{'name': 'neo4j', 'type': 'standard', 'aliases': [], 'access': 'read-write', 'address': 'p-c343b019-b149-0001.production-orch-1425.neo4j.io:7687', 'role': 'primary', 'writer': True, 'requestedStatus': 'online', 'currentStatus': 'online', 'statusMessage': '', 'constituents': []}


## Step 5 — เก็บชื่อ Database

In [ ]:
DATABASE = result.records[0]["name"]
print("Database =", DATABASE)


Database = neo4j


## Step 6 — ทดสอบการ Query

In [ ]:
result = driver.execute_query(
    '''
    RETURN "Hello Neo4j Aura" AS message
    ''',
    database_=DATABASE
)

print(result.records[0]["message"])


Hello Neo4j Aura


## Step 7 — ตรวจสอบจำนวน Node ที่มีอยู่

In [ ]:
result = driver.execute_query(
    '''
    MATCH (n)
    RETURN count(n) AS total_nodes
    ''',
    database_=DATABASE
)

print("จำนวน Node =", result.records[0]["total_nodes"])


จำนวน Node = 20


## Step 8 — ออกแบบ Graph ของ Food Recommender

จากไฟล์ Food Recommender เดิม ใช้ Node 2 ประเภท

- `User`
- `Food`

และ Relationship

- `(User)-[:LIKES]->(Food)`

ตัวอย่าง:
`(John)-[:LIKES]->(noodle)`


## Step 9 — สร้าง Constraint

In [ ]:
driver.execute_query(
    '''
    CREATE CONSTRAINT user_id_unique
    IF NOT EXISTS
    FOR (u:User)
    REQUIRE u.user_id IS UNIQUE
    ''',
    database_=DATABASE
)

driver.execute_query(
    '''
    CREATE CONSTRAINT food_id_unique
    IF NOT EXISTS
    FOR (f:Food)
    REQUIRE f.food_id IS UNIQUE
    ''',
    database_=DATABASE
)

print("สร้าง User และ Food constraint สำเร็จ")


สร้าง User และ Food constraint สำเร็จ


## Step 10 — เตรียมข้อมูล User และ Food

ข้อมูลชุดนี้มาจาก `FoodsRecommedersystem (1)`

In [ ]:
users = [
    {"user_id": "U001", "name": "John"},
    {"user_id": "U002", "name": "Billy"},
    {"user_id": "U003", "name": "Chanom"},
    {"user_id": "U004", "name": "Pop"},
    {"user_id": "U005", "name": "Alex"},
    {"user_id": "U006", "name": "Sarah"},
    {"user_id": "U007", "name": "David"},
    {"user_id": "U008", "name": "Emma"},
    {"user_id": "U009", "name": "Michael"},
    {"user_id": "U010", "name": "Sophia"}
]

foods = [
    {"food_id": "F001", "name": "noodle"},
    {"food_id": "F002", "name": "kaolao"},
    {"food_id": "F003", "name": "Kapawmoosaab"},
    {"food_id": "F004", "name": "fried rice"},
    {"food_id": "F005", "name": "Rad Na"},
    {"food_id": "F006", "name": "Pad Thai"},
    {"food_id": "F007", "name": "Tom Yum Goong"},
    {"food_id": "F008", "name": "Som Tum"},
    {"food_id": "F009", "name": "Green Curry"},
    {"food_id": "F010", "name": "Kao Mun Gai"}
]

print("Users =", len(users))
print("Foods =", len(foods))


Users = 10
Foods = 10


## Step 11 — สร้าง User ใน Neo4j Aura

In [ ]:
result = driver.execute_query(
    '''
    UNWIND $users AS row

    MERGE (u:User {
        user_id: row.user_id
    })

    SET u.name = row.name

    RETURN count(u) AS total
    ''',
    users=users,
    database_=DATABASE
)

print("จำนวน User ที่ประมวลผล =", result.records[0]["total"])


จำนวน User ที่ประมวลผล = 10


## Step 12 — สร้าง Food ใน Neo4j Aura

In [ ]:
result = driver.execute_query(
    '''
    UNWIND $foods AS row

    MERGE (f:Food {
        food_id: row.food_id
    })

    SET f.name = row.name

    RETURN count(f) AS total
    ''',
    foods=foods,
    database_=DATABASE
)

print("จำนวน Food ที่ประมวลผล =", result.records[0]["total"])


จำนวน Food ที่ประมวลผล = 10


## Step 13 — ดู User และ Food ทั้งหมด

In [ ]:
result = driver.execute_query(
    '''
    MATCH (n)
    WHERE n:User OR n:Food

    RETURN
        labels(n) AS type,
        properties(n) AS data
    ORDER BY type, data.name
    ''',
    database_=DATABASE
)

for record in result.records:
    print(record.data())


{'type': ['Food'], 'data': {'name': 'Green Curry', 'food_id': 'F009'}}
{'type': ['Food'], 'data': {'name': 'Kao Mun Gai', 'food_id': 'F010'}}
{'type': ['Food'], 'data': {'name': 'Kapawmoosaab', 'food_id': 'F003'}}
{'type': ['Food'], 'data': {'name': 'Pad Thai', 'food_id': 'F006'}}
{'type': ['Food'], 'data': {'name': 'Rad Na', 'food_id': 'F005'}}
{'type': ['Food'], 'data': {'name': 'Som Tum', 'food_id': 'F008'}}
{'type': ['Food'], 'data': {'name': 'Tom Yum Goong', 'food_id': 'F007'}}
{'type': ['Food'], 'data': {'name': 'fried rice', 'food_id': 'F004'}}
{'type': ['Food'], 'data': {'name': 'kaolao', 'food_id': 'F002'}}
{'type': ['Food'], 'data': {'name': 'noodle', 'food_id': 'F001'}}
{'type': ['User'], 'data': {'user_id': 'U005', 'name': 'Alex'}}
{'type': ['User'], 'data': {'user_id': 'U002', 'name': 'Billy'}}
{'type': ['User'], 'data': {'user_id': 'U003', 'name': 'Chanom'}}
{'type': ['User'], 'data': {'user_id': 'U007', 'name': 'David'}}
{'type': ['User'], 'data': {'user_id': 'U008', 'na

## Step 14 — เตรียม Relationship LIKES

ข้อมูลความชอบมาจากไฟล์เดิม เช่น

- John → noodle, kaolao
- Billy → noodle, kaolao, fried rice
- Chanom → noodle, Kapawmoosaab
- Pop → fried rice, Rad Na

รวมถึงข้อมูลของ Alex, Sarah, David, Emma, Michael และ Sophia ตามไฟล์เดิม


In [ ]:
likes = [
    {"user_id": "U001", "food_id": "F001"},
    {"user_id": "U001", "food_id": "F002"},

    {"user_id": "U002", "food_id": "F001"},
    {"user_id": "U002", "food_id": "F002"},
    {"user_id": "U002", "food_id": "F004"},

    {"user_id": "U003", "food_id": "F001"},
    {"user_id": "U003", "food_id": "F003"},

    {"user_id": "U004", "food_id": "F004"},
    {"user_id": "U004", "food_id": "F005"},

    {"user_id": "U005", "food_id": "F006"},
    {"user_id": "U005", "food_id": "F007"},

    {"user_id": "U006", "food_id": "F008"},
    {"user_id": "U006", "food_id": "F009"},

    {"user_id": "U007", "food_id": "F010"},
    {"user_id": "U007", "food_id": "F004"},

    {"user_id": "U008", "food_id": "F006"},
    {"user_id": "U008", "food_id": "F001"},

    {"user_id": "U009", "food_id": "F007"},
    {"user_id": "U009", "food_id": "F003"},

    {"user_id": "U010", "food_id": "F008"},
    {"user_id": "U010", "food_id": "F005"}
]

print("จำนวนความชอบ =", len(likes))


จำนวนความชอบ = 21


## Step 15 — สร้าง Relationship LIKES

In [ ]:
driver.execute_query(
    '''
    UNWIND $likes AS row

    MATCH
        (u:User {user_id: row.user_id}),
        (f:Food {food_id: row.food_id})

    MERGE (u)-[:LIKES]->(f)
    ''',
    likes=likes,
    database_=DATABASE
)

print("สร้าง LIKES สำเร็จ")


สร้าง LIKES สำเร็จ


## Step 16 — ดูว่าใครชอบอาหารอะไร

In [ ]:
result = driver.execute_query(
    '''
    MATCH
        (u:User)-[:LIKES]->(f:Food)

    RETURN
        u.user_id AS user_id,
        u.name AS user,
        f.food_id AS food_id,
        f.name AS food

    ORDER BY user_id, food
    ''',
    database_=DATABASE
)

for record in result.records:
    print(
        record["user"],
        "->",
        record["food"]
    )


John -> kaolao
John -> noodle
Billy -> fried rice
Billy -> kaolao
Billy -> noodle
Chanom -> Kapawmoosaab
Chanom -> noodle
Pop -> Rad Na
Pop -> fried rice
Alex -> Pad Thai
Alex -> Tom Yum Goong
Sarah -> Green Curry
Sarah -> Som Tum
David -> Kao Mun Gai
David -> fried rice
Emma -> Pad Thai
Emma -> noodle
Michael -> Kapawmoosaab
Michael -> Tom Yum Goong
Sophia -> Rad Na
Sophia -> Som Tum


## Step 17 — แสดงข้อมูลเป็นตาราง Pandas

In [ ]:
result = driver.execute_query(
    '''
    MATCH
        (u:User)-[:LIKES]->(f:Food)

    RETURN
        u.user_id AS user_id,
        u.name AS user,
        f.food_id AS food_id,
        f.name AS food

    ORDER BY user_id, food
    ''',
    database_=DATABASE
)

df = pd.DataFrame([record.data() for record in result.records])
df


,user_id,user,food_id,food
0,U001,John,F002,kaolao
1,U001,John,F001,noodle
2,U002,Billy,F004,fried rice
3,U002,Billy,F002,kaolao
4,U002,Billy,F001,noodle
5,U003,Chanom,F003,Kapawmoosaab
6,U003,Chanom,F001,noodle
7,U004,Pop,F005,Rad Na
8,U004,Pop,F004,fried rice
9,U005,Alex,F006,Pad Thai


## Step 18 — Traversal หลายทอด

โจทย์:
**John ชอบอาหารอะไร และมี User คนไหนที่ชอบอาหารเหมือน John?**

ใช้ Graph เดินจาก

`John → LIKES → Food ← LIKES ← User`


In [ ]:
result = driver.execute_query(
    '''
    MATCH
        (me:User {user_id: $user_id})
        -[:LIKES]->
        (food:Food)
        <-[:LIKES]-
        (other:User)

    WHERE other <> me

    RETURN
        me.name AS me,
        food.name AS shared_food,
        other.name AS similar_user

    ORDER BY similar_user, shared_food
    ''',
    user_id="U001",
    database_=DATABASE
)

for record in result.records:
    print(
        record["me"],
        "-> อาหารร่วม:",
        record["shared_food"],
        "-> User ที่คล้าย:",
        record["similar_user"]
    )


John -> อาหารร่วม: kaolao -> User ที่คล้าย: Billy
John -> อาหารร่วม: noodle -> User ที่คล้าย: Billy
John -> อาหารร่วม: noodle -> User ที่คล้าย: Chanom
John -> อาหารร่วม: noodle -> User ที่คล้าย: Emma


## Step 19 — Aggregation

โจทย์:
**แต่ละ User ชอบอาหารกี่อย่าง?**


In [ ]:
result = driver.execute_query(
    '''
    MATCH
        (u:User)-[:LIKES]->(f:Food)

    RETURN
        u.name AS user,
        count(f) AS total_foods

    ORDER BY total_foods DESC, user
    ''',
    database_=DATABASE
)

for record in result.records:
    print(
        record["user"],
        "=",
        record["total_foods"],
        "อย่าง"
    )


Billy = 3 อย่าง
Alex = 2 อย่าง
Chanom = 2 อย่าง
David = 2 อย่าง
Emma = 2 อย่าง
John = 2 อย่าง
Michael = 2 อย่าง
Pop = 2 อย่าง
Sarah = 2 อย่าง
Sophia = 2 อย่าง


## Step 20 — อาหารไหนได้รับความนิยมมากที่สุด

In [ ]:
result = driver.execute_query(
    '''
    MATCH
        (:User)-[:LIKES]->(f:Food)

    RETURN
        f.name AS food,
        count(*) AS like_count

    ORDER BY like_count DESC, food
    ''',
    database_=DATABASE
)

pd.DataFrame([record.data() for record in result.records])


,food,like_count
0,noodle,4
1,fried rice,3
2,Kapawmoosaab,2
3,Pad Thai,2
4,Rad Na,2
5,Som Tum,2
6,Tom Yum Goong,2
7,kaolao,2
8,Green Curry,1
9,Kao Mun Gai,1


## Step 21 — Mini Project: Recommendation System

โจทย์สำคัญ:

**แนะนำอาหารให้ John จากอาหารที่ User ที่มีความชอบร่วมกับ John ชอบ แต่ John ยังไม่ได้ชอบ**

หลักการ:
1. หาอาหารที่ John ชอบ
2. หา User คนอื่นที่ชอบอาหารเดียวกัน
3. ดูอาหารอื่นที่ User เหล่านั้นชอบ
4. ตัดอาหารที่ John ชอบอยู่แล้วออก
5. นับจำนวน User ที่สนับสนุนอาหารแต่ละรายการเป็น `score`


In [ ]:
result = driver.execute_query(
    '''
    MATCH
        (me:User {user_id: $user_id})
        -[:LIKES]->
        (shared:Food)
        <-[:LIKES]-
        (similar:User)
        -[:LIKES]->
        (food:Food)

    WHERE similar <> me
      AND NOT (me)-[:LIKES]->(food)

    RETURN
        food.food_id AS food_id,
        food.name AS recommendation,
        count(DISTINCT similar) AS score

    ORDER BY
        score DESC,
        recommendation
    ''',

    user_id="U001",
    database_=DATABASE
)

recommend_df = pd.DataFrame(
    [record.data() for record in result.records]
)

recommend_df


,food_id,recommendation,score
0,F003,Kapawmoosaab,1
1,F006,Pad Thai,1
2,F004,fried rice,1


## Step 22 — ทำ Recommendation เป็น Python Function

In [ ]:
def recommend_foods(user_id):

    query = '''
    MATCH
        (me:User {user_id: $user_id})
        -[:LIKES]->
        (shared:Food)
        <-[:LIKES]-
        (similar:User)
        -[:LIKES]->
        (food:Food)

    WHERE similar <> me
      AND NOT (me)-[:LIKES]->(food)

    RETURN
        food.food_id AS food_id,
        food.name AS recommendation,
        count(DISTINCT similar) AS score

    ORDER BY
        score DESC,
        recommendation
    '''

    result = driver.execute_query(
        query,
        user_id=user_id,
        database_=DATABASE
    )

    return pd.DataFrame(
        [record.data() for record in result.records]
    )


In [ ]:
recommend_foods("U001")


,food_id,recommendation,score
0,F003,Kapawmoosaab,1
1,F006,Pad Thai,1
2,F004,fried rice,1


## Step 23 — Function ดูข้อมูล User

In [ ]:
def find_user(user_id):

    result = driver.execute_query(
        '''
        MATCH (u:User {user_id: $user_id})

        RETURN
            u.user_id AS id,
            u.name AS name
        ''',
        user_id=user_id,
        database_=DATABASE
    )

    if len(result.records) == 0:
        return "ไม่พบ User"

    return result.records[0].data()


In [ ]:
find_user("U001")

{'id': 'U001', 'name': 'John'}

## Step 24 — Function ดูอาหารที่ User ชอบ

In [ ]:
def liked_foods(user_id):

    result = driver.execute_query(
        '''
        MATCH
            (u:User {user_id: $user_id})
            -[:LIKES]->
            (f:Food)

        RETURN
            f.food_id AS food_id,
            f.name AS food

        ORDER BY food
        ''',
        user_id=user_id,
        database_=DATABASE
    )

    return pd.DataFrame(
        [record.data() for record in result.records]
    )


In [ ]:
liked_foods("U001")

,food_id,food
0,F002,kaolao
1,F001,noodle


## Step 25 — ตรวจจำนวน Node และ Relationship

In [ ]:
result = driver.execute_query(
    '''
    MATCH (n)

    OPTIONAL MATCH (n)-[r]->()

    RETURN
        count(DISTINCT n) AS nodes,
        count(r) AS relationships
    ''',
    database_=DATABASE
)

print(result.records[0].data())


{'nodes': 20, 'relationships': 21}


## Step 26 — ดู Graph ใน Neo4j Aura

หลังจากรันข้อมูลจาก Colab แล้ว ให้กลับไปที่ **Neo4j Aura → Query** แล้วรัน:

```cypher
MATCH (n)-[r]->(m)
RETURN n, r, m
```

จะเห็น Graph เป็น

`User → LIKES → Food`

แทนโครงสร้าง Student → Book ใน Workshop เดิม


## Step 27 — ปิด Connection

เมื่อทำงานเสร็จแล้วค่อยรันเซลล์นี้


In [ ]:
driver.close()
print("Neo4j Aura connection closed")


Neo4j Aura connection closed
